In [1]:
import requests
from datetime import datetime, timedelta, timezone
from pathlib import Path
import json

BASE_URL = "https://api.energy-charts.info/price"
LOCATIONS = [
    "DE-LU",
    "AT",
    "BE",
    "CH",
    "CZ",
    "DK1",
    "DK2",
    "FR",
    "NL",
    "NO2",
    "PL",
    "SE4",
]

# Adjust these if you want a different window
START_DATE = datetime(2022, 1, 1, tzinfo=timezone.utc)
END_DATE = datetime.now(timezone.utc)
CHUNK_DAYS = 7  # shorter chunks reduce payload size per request

session = requests.Session()
session.headers.update({"Accept": "application/json"})

def fmt(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%dT%H:%MZ")

def fetch_price(bzn: str, start_dt: datetime, end_dt: datetime) -> dict:
    params = {"bzn": bzn, "start": fmt(start_dt), "end": fmt(end_dt)}
    resp = session.get(BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

def iter_chunks(start_dt: datetime, end_dt: datetime, days: int):
    cursor = start_dt
    while cursor < end_dt:
        next_cursor = min(cursor + timedelta(days=days), end_dt)
        yield cursor, next_cursor
        cursor = next_cursor

all_data = []

for bzn in LOCATIONS:
    for chunk_start, chunk_end in iter_chunks(START_DATE, END_DATE, CHUNK_DAYS):
        payload = fetch_price(bzn, chunk_start, chunk_end)
        all_data.append(
            {
                "bzn": bzn,
                "start": chunk_start.isoformat(),
                "end": chunk_end.isoformat(),
                "payload": payload,
            }
        )

# Example: write the collected data to disk once everything is fetched
output_path = Path("price-data-2022-today.json")
output_path.write_text(json.dumps(all_data))
print(f"Saved {len(all_data)} chunks to {output_path}")


HTTPError: 429 Client Error: Too Many Requests for url: https://api.energy-charts.info/price?bzn=DE-LU&start=2022-07-23T00%3A00Z&end=2022-07-30T00%3A00Z

In [ ]:
###old
HTTPAdapter
import requests
from requests.adapters import HTTPAdapter, Retry

url = "https://api.energy-charts.info/price"
params = {"bzn": "DE-LU", "start": "2023-01-01T17:55Z", "end": "2023-01-01T18:00Z"}

session = requests.Session()
session.headers.update({"Accept": "application/json"})
session.mount("https://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=0.3, status_forcelist=[500, 502, 503, 504])))

resp = session.get(url, params=params, timeout=10)
resp.raise_for_status()
data = resp.json()
print(data)
